# Т-Банк: поездки на самокатах

## Описание проекта

Сервис для аренды электросамокатов позволяет воспользоваться самокатом от Юрент через приложение Т-Банка. Для того, чтобы арендовать самокат, клиенту необходимо зайти в приложение, открыть сервис Самокаты и отсканировать QR код на руле самоката. В этот момент у клиента со счета списывается фиксированная сумма в качестве взноса. В конце поездки также списывается итоговая сумма за поездку.<br>
На первом этапе необходимо проанализировать данные, а затем сформировать ряд гипотез, которые могут стать основой для принятия решений по развитию бизнеса.

## Описание данных

Данные охватывают сезон аренды самокатов 2024 года (с апреля по октябрь 2024).<br>
Стоимость минуты и стоимость поездки измеряется в условных денежных единицах.<br>
<br>
Описание переменных:
- *order_rk* — Идентификатор заказа (поездки)
- *party_rk_id* — Идентификатор клиента
- *gender_cd* — Пол клиента
- *age* — Возраст клиента
- *education_level* — Уровень образования клиента:
  -  SCH — начальное, среднее
  -  GRD — высшее
  -  UGR — неполное высшее
  -  PGR — два высших
  -  ACD — ученая степень
- *marital_status_cd* — Семейный статус человека:
  -  IMR — состоит в незарегистрированном браке
  -  MAR —женат/замужем
  -  DLW — не проживает с супругом(ой)
  -  OMR — состоит в зарегистрированном браке
  -  CIV — гражданский брак
  -  UNM — холост/не замужем
  -  DIV — разведен(а)
  -  FAL — никогда не состоял(а) в браке
  -  WID — вдовец/вдова
- *lvn_state_nm* — Регион проживания человека
- *minute_cost* — Стоимость минуты
- *activation_cost* — Cтоимость активации
- *hold_amount* — Размер суммы, которая замораживается на счете в момент взятия самоката
- *transport_model* — Название модели самоката
- *distance_km* — Километраж поездки
- *created_dttm* — Дата и время создания заказа
- *book_start_dttm* —  Дата и время начала поездки
- *book_end_dttm* — Дата и время завершения поездки
- *book_time_zone_cd* — Часовой пояс
- *local_book_start_dttm* — Дата и время начала поездки в часовом поясе человека, который брал самокат
- *nominal_price_rub_amt* — Стоимость поездки
- *loyalty_accrual_rub_amt* — Размер выплаченного кэшбэка в рублях
- *loyalty_accrual_bns_amt* — Размер выплаченного кэшбэка в бонусах (если оплата происходила с помощью кредитной карты)

## План проекта

- Загрузка данных и изучение общей информации
- Предобработка данных
- Разведочный анализ данных (EDA)
- Общий вывод
- Презентация

## Загрузка данных и изучение общей информации

In [1]:
# импорт библиотек
# для анализа данных
import pandas as pd
import numpy as np
import datetime as dt

# для визуализации данных
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [2]:
import warnings
warnings.filterwarnings('ignore')

# снимаем ограничение на количество столбцов и строк
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# снимаем ограничение на ширину столбцов
pd.set_option('display.max_colwidth', None)

# выставляем ограничение на показ знаков после запятой
pd.options.display.float_format = '{:,.2f}'.format

Считаем данные из csv-файла в датафрейм:

In [3]:
try:
    df = pd.read_csv('C:/Users/888/OneDrive/Рабочий стол/TBank/kicksharing.csv')
except:
    df = pd.read_csv('https://dano.hse.ru/mirror/pubs/share/987942268.csv')

Выведем основную информацию о датафрейме:

In [4]:
# напишем функцию для обзора данных:
def print_primary_info(df):
    print("\nПервичная информация о датафрейме")
    print("Количество записей:", df.shape[0])
    print("Количество столбцов:", df.shape[1])
    print("\nПервые 10 строк:")
    display(df.head(10))
    print("Информация о типах данных: \n")
    print(df.info())
    print("\nПропуски:")
    display(df.isna().sum())
    print("\nПроцент пропусков от всего датафрейма:")
    display(df.isna().mean().sort_values(ascending=False)*100)
    print("\nДубликаты:")
    if df.duplicated().sum() > 0:
        print('Дубликатов: ', df.duplicated().sum())
    else:
        print('Дубликатов НЕТ')  

In [5]:
print_primary_info(df)


Первичная информация о датафрейме
Количество записей: 396749
Количество столбцов: 20

Первые 10 строк:


,order_rk,party_rk,gender_cd,age,education_level_cd,marital_status_cd,lvn_state_nm,minute_cost,activation_cost,hold_amount,transport_model,distance_km,created_dttm,book_start_dttm,book_end_dttm,book_time_zone_cd,local_book_start_dttm,nominal_price_rub_amt,loyalty_accrual_rub_amt,loyalty_accrual_bns_amt
0,266071307,761067705,M,40,UGR,DIV,ТЮМЕНСКАЯ ОБЛ,4.99,30.00,300.00,SL,3.69,2024-08-07 09:47:25.000000,2024-08-07 09:47:29.325252,2024-08-07 10:07:59.339524,5,2024-08-07 11:47:29.325252,134.79,19.48,NaN
1,355113920,614049469,F,30,GRD,MAR,РОСТОВСКАЯ ОБЛ,8.49,50.00,300.00,SL,1.11,2024-10-17 14:57:20.000000,2024-10-17 14:57:24.586000,2024-10-17 15:04:19.419607,3,2024-10-17 14:57:24.586000,109.43,40.30,NaN
2,347424551,757583701,M,28,UGR,UNM,СВЕРДЛОВСКАЯ ОБЛ,5.99,30.00,300.00,E,1.52,2024-09-19 05:31:41.000000,2024-09-19 05:34:59.476000,2024-09-19 05:41:50.164372,5,2024-09-19 07:34:59.476000,71.93,10.79,NaN
3,351562959,541367366,M,24,GRD,UNM,Г МОСКВА,8.99,50.00,300.00,SL,0.50,2024-10-04 16:05:09.000000,2024-10-04 16:05:13.162000,2024-10-04 16:07:31.724918,3,2024-10-04 16:05:13.162000,76.97,7.70,NaN
4,258647149,238473615,M,34,NaN,NaN,Г МОСКВА,6.99,50.00,300.00,SL,2.60,2024-07-10 06:57:40.000000,2024-07-10 06:57:43.017125,2024-07-10 07:07:48.446462,3,2024-07-10 06:57:43.017125,126.89,25.38,NaN
5,277397094,5247768,M,42,NaN,NaN,МОСКВА,7.49,50.00,300.00,SL,0.02,2024-09-10 09:40:18.000000,2024-09-10 09:40:24.036000,2024-09-10 09:41:03.644495,3,2024-09-10 09:40:24.036000,0.00,NaN,NaN
6,273528957,801272780,F,19,NaN,NaN,РЕСП ТАТАРСТАН,7.49,30.00,300.00,SL,4.09,2024-09-03 14:10:29.000000,2024-09-03 14:10:34.524349,2024-09-03 14:21:25.119585,3,2024-09-03 14:10:34.524349,112.39,28.00,NaN
7,265062394,866880584,M,38,NaN,NaN,СВЕРДЛОВСКАЯ ОБЛ,5.99,30.00,300.00,E,2.37,2024-08-03 03:09:35.000000,2024-08-03 03:09:38.493851,2024-08-03 03:18:47.841493,5,2024-08-03 05:09:38.493851,89.90,12.99,NaN
8,351562880,762053500,M,19,NaN,NaN,НОВОСИБИРСКАЯ ОБЛ,5.99,30.00,300.00,E,0.10,2024-10-04 10:45:16.000000,2024-10-04 10:45:20.457000,2024-10-04 10:47:39.863967,7,2024-10-04 14:45:20.457000,47.97,4.80,NaN
9,269365210,161703813,M,31,UGR,MAR,ЧЕЛЯБИНСКАЯ ОБЛ,6.49,30.00,300.00,E,1.49,2024-08-19 14:35:17.000000,2024-08-19 14:35:20.199325,2024-08-19 14:42:53.835731,5,2024-08-19 16:35:20.199325,81.92,81.92,NaN


Информация о типах данных: 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 396749 entries, 0 to 396748
Data columns (total 20 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   order_rk                 396749 non-null  int64  
 1   party_rk                 396749 non-null  int64  
 2   gender_cd                393828 non-null  object 
 3   age                      396749 non-null  int64  
 4   education_level_cd       190059 non-null  object 
 5   marital_status_cd        217729 non-null  object 
 6   lvn_state_nm             362572 non-null  object 
 7   minute_cost              396749 non-null  float64
 8   activation_cost          396749 non-null  float64
 9   hold_amount              396749 non-null  float64
 10  transport_model          396749 non-null  object 
 11  distance_km              396749 non-null  float64
 12  created_dttm             396749 non-null  object 
 13  book_start_dttm          39674

order_rk                        0
party_rk                        0
gender_cd                    2921
age                             0
education_level_cd         206690
marital_status_cd          179020
lvn_state_nm                34177
minute_cost                     0
activation_cost                 0
hold_amount                     0
transport_model                 0
distance_km                     0
created_dttm                    0
book_start_dttm                 0
book_end_dttm                   0
book_time_zone_cd               0
local_book_start_dttm           0
nominal_price_rub_amt           0
loyalty_accrual_rub_amt     29757
loyalty_accrual_bns_amt    374801
dtype: int64


Процент пропусков от всего датафрейма:


loyalty_accrual_bns_amt   94.47
education_level_cd        52.10
marital_status_cd         45.12
lvn_state_nm               8.61
loyalty_accrual_rub_amt    7.50
gender_cd                  0.74
created_dttm               0.00
nominal_price_rub_amt      0.00
local_book_start_dttm      0.00
book_time_zone_cd          0.00
book_end_dttm              0.00
book_start_dttm            0.00
order_rk                   0.00
distance_km                0.00
party_rk                   0.00
hold_amount                0.00
activation_cost            0.00
minute_cost                0.00
age                        0.00
transport_model            0.00
dtype: float64


Дубликаты:
Дубликатов НЕТ


### Промежуточный вывод

- В датафрейме 396748 записи в 20 столбцах.
- В столбцах *created_dttm*, *book_start_dttm*, *book_end_dttm*, *local_book_start_dttm* неоходимо заменить тип данных.
- В столбцах *gender_cd*, *education_level_cd*, *marital_status_cd*, *lvn_state_nm*, *loyalty_accrual_rub_amt*, *loyalty_accrual_bns_amt* имеются пропуски.
- Явных дубликатов и нет.

## Предобработка данных

### Замена типа данных

In [6]:
df[['created_dttm','book_start_dttm','book_end_dttm','local_book_start_dttm']] =  df[['created_dttm',\
                                                                                      'book_start_dttm',\
                                                                                      'book_end_dttm',\
                                                                                      'local_book_start_dttm']].astype('datetime64[s]')
print(df.dtypes)
print(df[['created_dttm','book_start_dttm','book_end_dttm','local_book_start_dttm']].head(5))

order_rk                           int64
party_rk                           int64
gender_cd                         object
age                                int64
education_level_cd                object
marital_status_cd                 object
lvn_state_nm                      object
minute_cost                      float64
activation_cost                  float64
hold_amount                      float64
transport_model                   object
distance_km                      float64
created_dttm               datetime64[s]
book_start_dttm            datetime64[s]
book_end_dttm              datetime64[s]
book_time_zone_cd                  int64
local_book_start_dttm      datetime64[s]
nominal_price_rub_amt            float64
loyalty_accrual_rub_amt          float64
loyalty_accrual_bns_amt          float64
dtype: object
         created_dttm     book_start_dttm       book_end_dttm  \
0 2024-08-07 09:47:25 2024-08-07 09:47:29 2024-08-07 10:07:59   
1 2024-10-17 14:57:20 2024-10-17 14:

### Устранение пропусков

В столбцах *loyalty_accrual_rub_amt* и *loyalty_accrual_bns_am* пропуски заменим на значение 0:

In [7]:
df['loyalty_accrual_rub_amt'] = df['loyalty_accrual_rub_amt'].fillna(0)
df['loyalty_accrual_bns_amt'] = df['loyalty_accrual_bns_amt'].fillna(0)

In [8]:
display(df.isna().sum())

order_rk                        0
party_rk                        0
gender_cd                    2921
age                             0
education_level_cd         206690
marital_status_cd          179020
lvn_state_nm                34177
minute_cost                     0
activation_cost                 0
hold_amount                     0
transport_model                 0
distance_km                     0
created_dttm                    0
book_start_dttm                 0
book_end_dttm                   0
book_time_zone_cd               0
local_book_start_dttm           0
nominal_price_rub_amt           0
loyalty_accrual_rub_amt         0
loyalty_accrual_bns_amt         0
dtype: int64

Пропуски в остальных столбцах заполнить нечем, оставим без изменений.

### Проверим уникальные значения в столбце *lvn_state_nm*:

In [9]:
print('Количество уникальных значений:', df['lvn_state_nm'].nunique())
df['lvn_state_nm'].sort_values().unique()

Количество уникальных значений: 212


array(['ISRAEL', 'АЛТАЙСКИЙ КРАЙ', 'АМУРСКАЯ ОБЛ',
       'АО ХАНТЫ-МАНСИЙСКИЙ АВТОНОМНЫЙ ОКРУГ - ЮГРА', 'АО ЯМАЛО-НЕНЕЦКИЙ',
       'АОБЛ ЕВРЕЙСКАЯ', 'АРХАНГЕЛЬСКАЯ ОБЛ', 'АСТРАХАНСКАЯ ОБЛ',
       'АСТРАХАНСКАЯ ОБЛАСТЬ', 'БАШКОРТОСТАН РЕСП', 'БЕЛГОРОДСКАЯ ОБЛ',
       'БРЕСТСКАЯ ОБЛАСТЬ', 'БРЯНСКАЯ ОБЛ', 'ВЛАДИМИРСКАЯ ОБЛ',
       'ВОЛГОГРАДСКАЯ ОБЛ', 'ВОЛГОГРАДСКАЯ ОБЛАСТЬ', 'ВОЛОГОДСКАЯ ОБЛ',
       'ВОРОНЕЖСКАЯ ОБЛ', 'Г БАЙКОНУР', 'Г МОСКВА', 'Г САМАРА',
       'Г САНКТ-ПЕТЕРБУРГ', 'Г СЕВАСТОПОЛЬ', 'Г. ЗЕЛЕНОГРАД', 'Г. МОСКВА',
       'Г. САНКТ-ПЕТЕРБУРГ', 'ГОМЕЛЬСКАЯ ОБЛ', 'ГОМЕЛЬСКАЯ ОБЛАСТЬ',
       'ГОРОД МОСКВА', 'ГОРОД САНКТ-ПЕТЕРБУРГ', 'ДОНЕЦКАЯ НАРОДНАЯ РЕСП',
       'ЕВРЕЙСКАЯ АОБЛ', 'ЗАБАЙКАЛЬСКИЙ КРАЙ', 'ЗАПОРОЖСКАЯ ОБЛ',
       'ИВАНОВСКАЯ ОБЛ', 'ИРКУТСКАЯ ОБЛ', 'ИРКУТСКАЯ ОБЛАСТЬ',
       'КАЛИНИНГРАДСКАЯ ОБЛ', 'КАЛУЖСКАЯ ОБЛ', 'КАМЧАТСКИЙ КРАЙ',
       'КЕМЕРОВСКАЯ ОБЛ', 'КЕМЕРОВСКАЯ ОБЛАСТЬ - КУЗБАСС',
       'КЕМЕРОВСКАЯ ОБЛАСТЬ - КУЗБАСС ОБЛ', 'КИРОВСКАЯ ОБЛ',
   

В списке уникальных значений есть много повторов с одинаковым названиями, но записанных по разному. Исправим.

In [ ]:
df['lvn_state_nm'] = (
    df['lvn_state_nm']
    .replace([' АОБЛ','АОБЛ ',' КРАЙ','КРАЙ ',' ОБЛАСТЬ',' ОБЛ','ОБЛ ','ОБЛ. ','ОБЛАСТЬ ',
              'АО ',' АО',' АВТОНОМНЫЙ ОКРУГ - ЮГРА','Г ','Г. ',' Г','ГОРОД ',
              ' ОБЛАСТЬ - КУЗБАСС',' - АЛАНИЯ','РЕСП ','РЕСПУБЛИКА ',' РЕСП','РЕСП. ',
              ' - КУЗБАСС',' АВТОНОМНЫЙ ОКРУГ'],'',regex=True)
)

In [ ]:
print('Количество уникальных значений:', df['lvn_state_nm'].nunique())
df['lvn_state_nm'].sort_values().unique()

In [ ]:
df['lvn_state_nm'] = (
    df['lvn_state_nm']
    .replace({'ЛЕНЕНГРАДСКАЯ':'ЛЕНИНГРАДСКАЯ', 'ЛНР':'ЛУГАНСКАЯ НАРОДНАЯ', 'МО ЗАПАД':'МОСКОВСКАЯ',
             'МОГИЛЁВСКАЯ':'МОГИЛЕВСКАЯ', 'МОСККОВСКАЯ':'МОСКОВСКАЯ', 'МОСКОВСКАЯ.':'МОСКОВСКАЯ',
             'НЕНЕЦКИЙ':'ЯМАЛО-НЕНЕЦКИЙ', 'ПОСЕЛЕНИЕ СОСЕНСКОЕ':'МОСКВА', 'РУСП ТАТАРСТАН':'ТАТАРСТАН',
             'САМАРА':'САМАРСКАЯ', 'ТАТАРСТАН (ТАТАРСТАН)':'ТАТАРСТАН', 'ЧЕЛЯБИНСКАЯ.':'ЧЕЛЯБИНСКАЯ',
             'ЧУВАШИЯ ЧУВАШСКАЯ -':'ЧУВАШИЯ', 'ЧУВАШСКАЯ - ЧУВАШИЯ':'ЧУВАШИЯ', 
              'ЧУВАШСКАЯ ЧУВАШИЯ':'ЧУВАШИЯ', 'ЗЕЛЕНОГРАД':'МОСКОВСКАЯ'})
)

In [ ]:
print('Количество уникальных значений:', df['lvn_state_nm'].nunique())
df['lvn_state_nm'].sort_values().unique()

In [ ]:
df.groupby('lvn_state_nm').agg({'order_rk' : 'count'}).reset_index()

Устранение дубликатов в наименованиях регионов позволило сократить их число с 212 до 98.

### Добавление столбцов

#### Время поездки

In [ ]:
df['trip_time_minut'] = (df['book_end_dttm'] - df['book_start_dttm']).dt.total_seconds() / 60

#### Полученные бонусы

Создадим отдельный столбец с данными о полученных бонусах.

In [ ]:
df['bonus'] = df['loyalty_accrual_rub_amt'] + df['loyalty_accrual_bns_amt']

In [ ]:
display(df.head(3))

#### Месяц, день недели и час поездки

Для начало перепроверим период даных.

In [ ]:
df['local_book_start_dttm'].dt.year.unique()

Месяц поездки:

In [ ]:
df['month_trip'] = df['local_book_start_dttm'].dt.month

In [ ]:
df['month_trip'].unique()

День недели поездки:

In [ ]:
df['weekday_trip'] = df['local_book_start_dttm'].dt.weekday
df['weekday_trip'].unique()

В какой час совершена поездка:

In [ ]:
df['hour_trip'] = df['local_book_start_dttm'].dt.hour
df['hour_trip'].unique()

### Удаление аномальных значений

Для дальнейшего использования скопируем новый датасет. <br>
Для уменьшения таблицы удалим из датасета столбцы: *created_dttm*, *book_start_dttm*, *book_end_dttm*, *book_time_zone_cd*, *loyalty_accrual_rub_amt*, *loyalty_accrual_bns_amt*, *transport_model*. Для дальнейшего анализа они не нужны.

In [ ]:
new_df = df.copy()
new_df = new_df.drop(['created_dttm',
                      'book_start_dttm', 
                      'book_end_dttm',
                      'book_time_zone_cd',
                      'loyalty_accrual_rub_amt',
                      'loyalty_accrual_bns_amt',
                      'transport_model'], axis=1)
display(new_df.head(5))

In [ ]:
display(new_df.describe().T)

В некоторых столбцах содержатся аномальные значения. Проверим по порядку столбцы с числовыми значениями.

#### Столбец *age*

In [ ]:
print(sorted(new_df['age'].unique()))
print("Количество пользователей до 18 лет:", new_df['order_rk'].loc[new_df['age']<18].count())

По правилам сервиса, арендовать самокат можно с 18 лет. На текущий момент таких пользователей до 18 лет - 47. <br>
Вероятно это ошибка в записях, удалим их.

In [ ]:
new_df = new_df.query('age >= 18')
print(sorted(new_df['age'].unique()))

#### Столбец *distance_km*

In [ ]:
print(len(new_df[new_df['distance_km'] == 0]))

Слишком много записей с километражем поездки равное 0, не похоже на ошибку. Вероятно это погрешность GPS-данных.

In [ ]:
fig, ax = plt.subplots(figsize = (20,2))
ax = new_df[['distance_km']]\
           .boxplot(vert = False, ax=ax)
ax.set_title('Диаграмма размаха километраж поездки ', pad=15, size=15)
ax.set_xlabel('Километры', labelpad=10, size=12)
plt.xlim(0,20)
plt.show()

In [ ]:
print(new_df['distance_km'].quantile(.99))

In [ ]:
print(len(new_df.query('distance_km >=15 & distance_km < 100')))
print(len(new_df[new_df['distance_km'] >= 15]))
print(len(new_df[new_df['distance_km'] >= 30]))

Наблюдается высокий процент заказов с пробегом ≥ 15 км. Один из заказов показал 56 012,64 км — явно аномальное значение (вероятна ошибка GPS или записи). Для анализа с использованием этого столбца ограничимся диапазоном 0–15 км.

#### Столбец *bonus*

In [ ]:
display(new_df[new_df['bonus'] < 0])

У 7 заказов отрицательный бонус, вероятно при оплате пользователь списал накопленные бонусы.

#### Столбец *lvn_state_nm*

В столбце *lvn_state_nm* есть регионы не относящиеся к РФ, удалим их.

In [ ]:
new_df.groupby('lvn_state_nm').agg({'order_rk' : 'count'}).sort_values(by='order_rk').head(15)

In [ ]:
new_df = new_df[~new_df['lvn_state_nm'].isin(['ПОЛЬША', 'ОДЕССКАЯ', 'МОГИЛЕВСКАЯ',\
                                              'НОВОЗЫБКОВСКАЯ', 'ISRAEL', 'ГОМЕЛЬСКАЯ',\
                                              'БАЙКОНУР', 'ХЕРСОНСКАЯ', 'ЗАПОРОЖСКАЯ',\
                                              'СОЕДИНЕННЫЕ ШТАТЫ АМЕРИКИ'])]
print('Количество уникальных значений:', new_df['lvn_state_nm'].nunique())
new_df.groupby('lvn_state_nm').agg({'order_rk' : 'count'}).sort_values(by='order_rk').head(10)

In [ ]:
print('Размер исходного датасета, кол-во записей:', len(df))
print('Размер нового датасета, кол-во записей:', len(new_df))
print(f"Разница в размерах: {(len(df)-len(new_df))/len(df)*100:.2f}%")

### Промежуточный вывод

- В столбцах с данными о времени поменяли тип на datetime64[s].
- В столбце *lvn_state_nm* исправили дублирующие названия.
- В столбцах *loyalty_accrual_rub_amt* и *loyalty_accrual_bns_amt* пропуски заполнили значением 0.
- Добавили дополнительные столбцы: *trip_time_minut*, *bonus*, *month_trip*, *weekday_trip* и *hour_trip*.
- Скопировали новый датасет *new_df* и удалили не нужные для анализа столбцы.
- Удалили данные о несовершенолетних пользователей. 
- Удалили данные по заказам не из РФ.
- Потери данных 0.03%

## Разведочный анализ данных (EDA)

### Сводная статистика

In [ ]:
display(new_df.describe().T)
print('Всего заказов(поездок) за период:', new_df['order_rk'].nunique())
print('Всего пользователей за период:', new_df['party_rk'].nunique())

**Вывод:** -Всего сервисом за период воспользовались 64002 пользователя и совершили 396649 заказа.

### Количество поездок на пользователя

In [ ]:
order_counts = (new_df.groupby('party_rk', as_index=False)
                  .agg({'order_rk': 'nunique'}))

display(order_counts.sort_values(by='order_rk', ascending=False).head(10))
display(order_counts['order_rk'].describe())

In [ ]:
one_trip = order_counts['order_rk'][order_counts['order_rk'] == 1].count()
print("Количество пользователь с одной поездкой за период:", one_trip)
print(f"Их доля от общего: {one_trip/64002*100:.2f} %")
two_trip = order_counts['order_rk'][order_counts['order_rk'] == 2].count()
print("Количество пользователь с двумя поездками за период:", two_trip)
print(f"Их доля от общего: {two_trip/64002*100:.2f} %")

In [ ]:
fig, ax = plt.subplots(figsize = (20,2))
ax = new_df.groupby('party_rk')\
           .agg({'order_rk':'nunique'})\
           .boxplot(vert = False, ax=ax)
ax.set_title('Диаграмма размаха количества заказов на клиента', pad=15, size=15)
ax.set_xlabel('Количество', labelpad=10, size=12)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize = (20,2))
ax = new_df.groupby('party_rk')\
          .agg({'order_rk':'nunique'})\
          .boxplot(vert = False, ax=ax)
ax.set_title('Диаграмма размаха количества заказов на клиента', pad=15, size=15)
ax.set_xlabel('Количество', labelpad=10, size=12)
plt.xlim(0,150)
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
plt.hist(order_counts['order_rk'], 
        bins=200,
        edgecolor='black',
        alpha=0.7)

plt.title('Распределение количества заказов на пользователя', fontsize=15, fontweight='bold')
plt.xlabel('Количество заказов на пользователя', fontsize=12)
plt.ylabel('Число пользователей', fontsize=12)

plt.grid(axis='y')
plt.xlim(0,50)
plt.tight_layout()
plt.show()

In [ ]:
order_counts.query('order_rk >= 20').count()

**Вывод:** За период 64 002 пользователя:

- большинство совершили меньше 10 заявок за период;

- за период 40,6% пользователей оформили 1 поездку, 16,9 - 2 поездки, 4 427 пользователей (6,9 %) оформили 20+ поездок;

- среднее число заявок на пользователя — 6, медиана — 2.

### Распределение поездок по месяцам за период

In [ ]:
new_df.pivot_table(index='month_trip', values='order_rk', aggfunc='count')

In [ ]:
#plt.figure(figsize=(20, 10))
new_df.pivot_table(index='month_trip', values='order_rk', aggfunc='count')\
      .plot(kind='bar', width=0.7, ec='black', legend = False)
plt.title('Распределение поездок по месяцам за период', pad=17, size=15, fontweight='bold')
plt.xlabel('Месяц', size=12)
month = ['Апрель', 'Май', 'Июнь', 'Июль', 'Август', 'Сентябрь', 'Октябрь']
plt.xticks(range(len(month)), month, rotation=1)
plt.ylabel('Количество поездок', size=12)
plt.grid(alpha=0.45)
plt.tight_layout()

plt.show()

**Вывод:** Основными месяцами по объёму поездок являются Июль, Август и Сентябрь.

### Распределение поездок по дням недели за период

In [ ]:
new_df.pivot_table(index='weekday_trip', values='order_rk', aggfunc='count')\
      .sort_values(by='weekday_trip')

In [ ]:
#plt.figure(figsize=(25, 10))
new_df.pivot_table(index='weekday_trip', values='order_rk', aggfunc='count')\
      .plot(kind='bar', width=0.7, ec='black', legend = False)
plt.title('Распределение поездок по дням недели за период', pad=17, size=15, fontweight='bold')
plt.xlabel('День недели', size=12)
days = ['Пн', 'Вт', 'Ср', 'Чт', 'Пт', 'Сб', 'Вс']
plt.xticks(range(len(days)), days, rotation=1)
plt.ylabel('Количество поездок', size=12)
plt.grid(alpha=0.45)
plt.tight_layout()

plt.show()

**Вывод:** С понедельника по пятницу число поездок растёт — пик приходится на пятницу; минимум поездок — в воскресенье.

### Распределение поездок по часам за период

In [ ]:
new_df.pivot_table(index='hour_trip', values='order_rk', aggfunc='count')

In [ ]:
#plt.figure(figsize=(20, 10))
new_df.pivot_table(index='hour_trip', values='order_rk', aggfunc='count')\
      .plot(kind='bar', width=0.7, ec='black', legend = False)
plt.title('Распределение поездок по часам за период', pad=17, size=15, fontweight='bold')
plt.xlabel('Часы', size=12)
plt.xticks(rotation=1)
plt.ylabel('Количество поездок', size=12)
plt.grid(alpha=0.45)
plt.tight_layout()

plt.show()

**Вывод:** Наиболее активное время поездок — с 4 до 20 часов. При этом фиксируются два пика активности: ранний (в 5 часов) и дневной (с 14 до 17 часов).

### Распределение поездок по полу пользователей

In [ ]:
new_df.pivot_table(index='gender_cd', values='order_rk', aggfunc='count')

In [ ]:
#plt.figure(figsize=(10, 10))
new_df.pivot_table(index='gender_cd', values='order_rk', aggfunc='count')\
      .plot(kind='bar', width=0.7, ec='black', legend = False)
plt.title('Распределение поездок по полу пользователей', pad=17, size=15, fontweight='bold')
plt.xlabel('Пол', size=12)
gender = ['Женский', 'Мужской']
plt.xticks(range(len(gender)), gender, rotation=1)
plt.ylabel('Количество поездок', size=12)
plt.grid(alpha=0.45)
plt.tight_layout()

plt.show()

**Вывод:** Среди пользователей сервиса мужчин в шесть раз больше, чем женщин.

### Распределение по возрасту пользователей

In [ ]:
new_df.pivot_table(index='age', values='order_rk', aggfunc='count')

In [ ]:
display(new_df['age'].describe())

In [ ]:
new_df.pivot_table(index='age', values='order_rk', aggfunc='count')\
      .plot(kind='bar', width=0.7, ec='black', legend = False, figsize=(25, 10))
plt.title('Распределение поездок по возрасту пользователей', pad=17, size=15, fontweight='bold')
plt.xlabel('Возраст', size=12)
plt.xticks(rotation=1)
plt.ylabel('Количество поездок', size=12)
plt.grid(alpha=0.45)

plt.show()

In [ ]:
print("Средний возраст:", new_df['age'].mean(),"лет.")
print("Медианный возраст:", new_df['age'].median(),"лет.")

**Вывод:** Сервис охватывает все возрастные категории, но большинство пользователей — в возрасте 18–40 лет. Средний возраст 31,5 лет и медианный возраст равен 31 году.

### Распределение по уровеню образования клиента

In [ ]:
new_df.pivot_table(index='education_level_cd', values='order_rk', aggfunc='count')\
      .sort_values(by='order_rk')

In [ ]:
new_df.pivot_table(index='education_level_cd', values='order_rk', aggfunc='count')\
      .sort_values(by='order_rk')\
      .plot(kind='bar', width=0.7, ec='black', legend = False)
plt.title('Распределение поездок по уровеню образования пользователей', pad=17, size=15, fontweight='bold')
plt.xlabel('Уровень образования', size=12)
plt.xticks(rotation=1)
plt.ylabel('Количество поездок', size=12)
plt.grid(alpha=0.45)
plt.tight_layout()
plt.show()

**Вывод:** Больше всего пользователей с высшим образованием.

### Распределение по cемейному статусу клиента

In [ ]:
new_df.pivot_table(index='marital_status_cd', values='order_rk', aggfunc='count')\
      .sort_values(by='order_rk')

**Вывод:** В основном клиенты холостые(не замужем).

### Распределение по региону

In [ ]:
new_df.pivot_table(index='lvn_state_nm', values='order_rk', aggfunc='count')\
      .sort_values(by='order_rk', ascending=False)

In [ ]:
new_df.pivot_table(index='lvn_state_nm', values='order_rk', aggfunc='count')\
      .sort_values(by='order_rk', ascending=False)\
      .head(10)\
      .plot(kind='bar', width=0.7, ec='black', legend = False, figsize=(15, 10))
plt.title('Распределение поездок по региону ТОП-10', pad=17, size=15, fontweight='bold')
plt.xlabel('Регион', size=12)
plt.xticks(rotation=1)
plt.ylabel('Количество поездок', size=12)
plt.grid(alpha=0.5)
plt.tight_layout()

plt.show()

**Вывод:** Основные регионы  — Москва и область, Санкт‑Петербург, Свердловская область, Краснодарский край.

### Распределение по расстоянию поездки

Для распределения возьмем срез 0-15 км.

In [ ]:
new_df_km = new_df[new_df['distance_km']<15]

new_df_km['distance_km'].hist(bins=150, figsize=(15, 8), ec='black', legend=True)
plt.title('Распределение по расстоянию поездки', size=20)
plt.xlabel('Расстояние', size=15)
plt.ylabel('Количество поездок', size=15)
plt.tight_layout()
plt.show()

In [ ]:
print("Средний пробег поездки:", round(new_df_km['distance_km'].mean(), 2),"км.")
print("Медианный пробег:", round(new_df_km['distance_km'].median(), 2),"км.")

**Вывод:** 
- Много заказов с пробегом около нуля. Здесь повидиму присутствует погрешность GPS.
- Основная масса заказов с пробегом до 3 км, средний пробег - 2.32км и медианный - 1.71км.

### Распределение поездок по продолжительности поедки

In [ ]:
new_df['trip_time_minut'].describe()

In [ ]:
fig, ax = plt.subplots(figsize = (20,2))
ax = new_df[['trip_time_minut']]\
           .boxplot(vert = False, ax=ax)
ax.set_title('Диаграмма размаха продолжительности поедки ', pad=15, size=15)
ax.set_xlabel('Продолжительность поедки, мин.', labelpad=10, size=12)

plt.show()

In [ ]:
display(new_df[new_df['trip_time_minut'] > 300])

In [ ]:
new_df['trip_time_minut'].hist(bins=500, figsize=(10, 6), ec='black', legend=True)
plt.title('Распределение поездок по продолжительности поедки', size=20)
plt.xlabel('Продолжительность поедки, мин.', size=15)
plt.ylabel('Количество поездок', size=15)
plt.tight_layout()
plt.xlim(0,80)
plt.show()

In [ ]:
print("Среднее время продолжительности поедки:", round(new_df['trip_time_minut'].mean(), 2),"мин.")
print("Медианное время продолжительности поедки:", round(new_df['trip_time_minut'].median(), 2),"мин.")

In [ ]:
print(new_df['trip_time_minut'].quantile(.80))

**Вывод:** В основном поездка длится до 15 минут (80% заказов). Среднее время 11.44 мин. и медианное - 7.77 мин.

### Сколько тратят на поездку

In [ ]:
fig, ax = plt.subplots(figsize = (20,2))
ax = new_df[['nominal_price_rub_amt']]\
           .boxplot(vert = False, ax=ax)
ax.set_title('Диаграмма размаха трат на поездку', pad=15, size=15)
ax.set_xlabel('Стоимость поездки', labelpad=10, size=12)

plt.show()

In [ ]:
new_df['nominal_price_rub_amt'].hist(bins=200, figsize=(10, 6), ec='black', legend=True)
plt.title('Сколько тратят на поездку', size=20)
plt.xlabel('Стоимость поездки', size=15)
plt.ylabel('Количество поездок', size=15)
plt.tight_layout()
plt.xlim(0,550)
plt.show()

In [ ]:
print("Средняя стоимость поездки:", round(new_df['nominal_price_rub_amt'].mean(), 2),"у.е.")
print("Медианная стоимость:", round(new_df['nominal_price_rub_amt'].median(), 2),"у.е.")

In [ ]:
print(new_df['nominal_price_rub_amt'].quantile(.95))

**Вывод:** 
- В основном пользователи сервиса тратят на поездку от 50 до 200 у.е.
- 5% заказов стоимостью больше 285 у.е.
- Средняя стоимость поездки 127.65 у.е и медианная стоимость: 103.94 у.е.

### Количество бонусов

In [ ]:
fig, ax = plt.subplots(figsize = (20,2))
ax = new_df[['bonus']]\
           .boxplot(vert = False, ax=ax)
ax.set_title('Диаграмма размаха бонусов на поездку', pad=15, size=15)
ax.set_xlabel('Количество бонусов', labelpad=10, size=12)

plt.show()

In [ ]:
print(new_df['bonus'].quantile(.95))
print(new_df['bonus'].quantile(.99))

In [ ]:
fig, ax = plt.subplots(figsize = (20,2))
ax = new_df[['bonus']]\
           .boxplot(vert = False, ax=ax)
ax.set_title('Диаграмма размаха бонусов на поездку', pad=15, size=15)
ax.set_xlabel('Количество бонусов', labelpad=10, size=12)
plt.xlim(0,400)
plt.show()

In [ ]:
new_df['bonus'].hist(bins=1000, figsize=(10, 6), ec='black', legend=True)
plt.title('Бонусы за поездку', size=20)
plt.xlabel('Количество бонусов', size=15)
plt.ylabel('Количество поездок', size=15)
plt.tight_layout()
plt.xlim(0,400)
plt.show()

In [ ]:
print("Среднее количество бонусов:", round(new_df['bonus'].mean(), 2))
print("Медианное количество:", round(new_df['bonus'].median(), 2))

**Выод:** 
- Пользователи получают за поездку до 50 бонусов
- 5% заказов получили больше 131 бонуса и 1% - больше 378.54. Есть заказы с совсем большими бонусами, вероятно ошибка записи.
- Среднее количество бонусов: 41.76 и медианное количество: 19.09.

### Матрица корреляции:

In [ ]:
param_corr = [
    'minute_cost',
    'activation_cost',
    'distance_km',
    'nominal_price_rub_amt',
    'trip_time_minut',
    'bonus'
]

Очистим данные от выбросов

In [ ]:
def quartile_range(data, column): # Функция для подсчёта границ с учетом квартилей
    q1 = data[column].quantile(.25)
    q3 = data[column].quantile(.75)
    iqr = q3 - q1
    dfq = data.loc[(data[column] < q3 + 2*iqr) & (data[column] > q1 - 2*iqr), column]
    return dfq
    
for col in param_corr:
    new_df[col] = quartile_range(new_df, col)

In [ ]:
# Матрица корреляции:
last_price_corr = new_df[param_corr].corr()
display(last_price_corr)

In [ ]:
plt.figure(figsize = (10, 8))
sns.heatmap(last_price_corr, vmin=-1, vmax=1, cmap='coolwarm', annot=True)
plt.title('Матрица корреляции', size=20)
#plt.tight_layout()
plt.show()

**Вывод:** 
- Присутствует сильная корреляция в парах: (nominal_price_rub_amt-trip_time_minut), (nominal_price_rub_amt-distance_km), (trip_time_minut-distance_km). Но это естественно.
- Так же столбец bonus имеет небольшую корреляцию в паре с nominal_price_rub_amt.

## Общий вывод

Анализ данных за период 2024 года показал следуещее:
- **Объём использования:** 64 002 пользователя, 396 649 заказов.

- **Активность пользователей:**

    - 40,6 % — 1 поездка, 16,9 % — 2 поездки, лишь 6,9 % — 20+ поездок;

    - среднее число заказов на пользователя — 6, медиана — 2.

    - Сезонность: пик поездок — июль–сентябрь.

- **Дни недели:** рост с понедельника по пятницу (пик в пятницу), минимум — в воскресенье.

- **Время суток:** активность 14:00–20:00, пики — в 05:00 и 14:00–17:00.

- **Пол:** мужчин в 6 раз больше, чем женщин.

- **Возраст:** преимущественно 18–40 лет (средний — 31,5 года, медианный — 31 год).

- **Образование:** преобладает высшее.

- **Семейное положение:** в основном холосты/не замужем.

- **География:** Москва и область, Санкт‑Петербург, Свердловская область, Краснодарский край.

- **Пробег:** до 3 км (средний — 2,32 км, медианный — 1,71 км).

- **Длительность поездки:** до 15 мин (средняя — 11,44 мин, медианная — 7,77 мин).

- **Стоимость:** 50–200 у. е. (средняя — 127,65 у. е., медианная — 103,94 у. е.); 5 % заказов > 285 у. е.

- **Бонусы:** до 50 бонусов за поездку (среднее — 41,76, медианное — 19,09); 5 % заказов — > 131 бонуса, 1 % — > 378,54.

**Портрет основного пользователя:** <br>
Молодой мужчина (около 31 года) с высшим образованием, не женат. Проживает в Москве/области, Санкт‑Петербурге, Свердловской области или Краснодарском крае. Использует сервис нерегулярно (чаще 1–2 раза за сезон), преимущественно в рабочие дни (пик — пятница) днём (14:00–17:00) или вечером (до 20-00). Совершает короткие поездки (до 3 км, до 15 минут), тратя на них 50–200 у. е. и и за поездку получает до 50 бонусов. 
<br>
<br>
**Итоговвый вывод:** <br>
- Анализ показал, что 80 % поездок длится менее 15 минут, а средний пробег составляет 2,3 км. Выявлены пики активности в пятницу и в часы 14:00–17:00.
- Обнаружены аномалии: часть заказов с большим пробегом и небольшой длительностью поездки или наоборот с нулевым пробегом (вероятно, ошибки GPS). Есть аномалии по полученным бонусам и по продолжительности поездки. Рекомендуется проверить геоданные и правильность заполнения данных.

## Гипотезы

1. Внедрение дифференцированных тарифов по дням недели повысит загрузку в «низкий» сезон (воскресенье) и увеличит выручку.
    - Суть: предложить сниженную стоимость поездки (например, минус 15-20 %) в воскресенье.
    - Ожидания: рост числа поездок в воскресенье.
2. Программа лояльности с прогрессивным кешбэком (больше поездок = выше процент) повысит удержание и средний чек.
    - Суть: ввести уровни кешбэка (к примеру 1–4 поездки/месяц = 5 % кешбэка; 5–9 поездок/месяц = 10 % кешбэка; 10+ поездок/месяц = 15 % кешбэка).
    - Ожидания: увеличение доли пользователей с 10+ поездок, рост среднего числа поездок на пользователя.

## Презентация

Ссылка на презентацию: https://disk.yandex.ru/i/nthbZX7KhPMP-w